In [ ]:
#Import Libraries
import os
import cv2
import numpy as np
import tifffile as tiff
import torch
import pandas as pd
from matplotlib import pyplot as plt
from multiprocessing import Pool, cpu_count
from PIL import Image
from torchvision import transforms
from skimage.feature import graycomatrix, graycoprops, local_binary_pattern
from skimage.measure import shannon_entropy
from scipy.stats import skew, kurtosis
from scipy.fftpack import fft2
import re

In [ ]:
#Processing Thermal bin to thermal tiff files to preserve information
def detect_best_scaling_factor(frame_matrix):
    """Automatically detects the best scale factor based on expected temperature range."""
    scale_factors = [10, 50, 100]  # Common values
    best_scale = None
    min_diff = float("inf")

    for scale in scale_factors:
        temp_matrix = (frame_matrix / scale) - 273.15
        min_temp, max_temp = np.min(temp_matrix), np.max(temp_matrix)

        # We expect temperatures to be between -10°C and 50°C for natural objects
        expected_range = (-10, 50)
        diff = abs(min_temp - expected_range[0]) + abs(max_temp - expected_range[1])

        if diff < min_diff:
            min_diff = diff
            best_scale = scale

    return best_scale

def read_tommy_bin_file(filename, first_frame=46, num_frames=None, skip_frames=46):
    """Reads TOMMY thermal bin files and processes frames based on MATLAB logic."""

    frames = []

    with open(filename, 'rb') as f:
        f.seek(0, 2)
        file_size = f.tell()
        f.seek(0)

        # Read frame dimensions
        frame_size = np.frombuffer(f.read(8), dtype=np.dtype('>u4'))
        height, width = frame_size
        print(f"Frame dimensions: {height}x{width}")

        frame_data_size = height * width * 2  # 2 bytes per pixel
        frame_total_size = frame_data_size + 8

        max_frames = file_size // frame_total_size
        print(f"Max frames available: {max_frames}")

        if num_frames is None:
            num_frames = (max_frames - first_frame) // (skip_frames + 1)

        if first_frame + num_frames * (skip_frames + 1) > max_frames:
            raise ValueError("Requested frames exceed maximum frames in the file.")

        f.seek(first_frame * frame_total_size, 0)

        for frame_idx in range(num_frames):
            header = np.frombuffer(f.read(8), dtype=np.dtype('>u4'))
            if header[0] != height or header[1] != width:
                print(f"Frame header mismatch at frame {frame_idx}. Skipping...")
                continue

            frame_data = np.frombuffer(f.read(frame_data_size), dtype=np.dtype('>u2'))
            if frame_data.size != height * width:
                print(f"Incomplete frame at index {frame_idx}. Stopping...")
                break

            # Reshape and flip the frame
            frame_matrix = frame_data.reshape((height, width))
            frame_matrix = np.flipud(np.fliplr(frame_matrix))

            # Detect best scale factor
            best_scale = detect_best_scaling_factor(frame_matrix)

            # Convert to Celsius using detected scale factor
            frame_matrix = (frame_matrix / best_scale) - 273.15

            frames.append(frame_matrix)

            if skip_frames > 0:
                f.seek(skip_frames * frame_total_size, 1)

    return frames

def save_frames(frames, output_folder):
    """Saves thermal frames in TIFF format (raw data) and JPG (colormap visualization)."""

    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    for idx, frame in enumerate(frames):
        flipped_frame = np.rot90(frame, k=3)

        vmin, vmax = np.min(flipped_frame), np.max(flipped_frame)

        # Save raw temperature data as float32 TIFF
        tiff_output_path = os.path.join(output_folder, f"frame_{idx + 1}.tiff")
        tifffile.imwrite(tiff_output_path, flipped_frame.astype(np.float32))

    print(f"Saved {len(frames)} frames to {output_folder}")

def process_all_folders(parent_folder):
    """Process all folders in the parent folder to extract and save thermal frames."""

    for root, dirs, files in os.walk(parent_folder):
        for dir_name in dirs:
            subfolder_path = os.path.join(root, dir_name)
            thermal_files = [f for f in os.listdir(subfolder_path) if f.endswith('.bin')]

            if not thermal_files:
                print(f"No .bin files found in {subfolder_path}")
                continue

            for thermal_file in thermal_files:
                output_folder = os.path.join(subfolder_path, os.path.splitext(thermal_file)[0] + "_tiff")

                try:
                    os.makedirs(output_folder, exist_ok=True)
                except OSError as e:
                    print(f"Error creating directory {output_folder}: {e}")
                    continue

                thermal_file_path = os.path.join(subfolder_path, thermal_file)
                print(f"Processing file: {thermal_file_path}")

                frames = read_tommy_bin_file(thermal_file_path)

                if not frames:
                    print(f"No frames extracted from {thermal_file_path}")
                    continue

                save_frames(frames, output_folder)
                print(f"Saved frames to {output_folder}")

# Usage
parent_folder = "/content/drive/MyDrive/Thesis/POM-IMG/Data-14.10.2024"
process_all_folders(parent_folder)


In [ ]:
#Crop RGB images to 4900 X 3300
def crop_and_save(image_path, output_path, x_start=0, x_end=None, y_start=0, y_end=None):
    # Load the image
    image = cv2.imread(image_path)
    if image is None:
        print(f"Error: Unable to load image {image_path}")
        return

    # Get image dimensions
    img_height, img_width = image.shape[:2]

    # Default x_end and y_end to the image dimensions if not specified
    if x_end is None:
        x_end = img_width
    if y_end is None:
        y_end = img_height

    # Perform the crop
    cropped_image = image[y_start:y_end, x_start:x_end]

    # Save the cropped image
    cv2.imwrite(output_path, cropped_image)
    print(f"Cropped image saved to: {output_path}")

def crop_images_in_parent_folder(parent_parent_folder, x_start=0, x_end=None, y_start=0, y_end=None):

    for parent_folder in os.listdir(parent_parent_folder):
        parent_folder_path = os.path.join(parent_parent_folder, parent_folder)
        if os.path.isdir(parent_folder_path):
            # Define input and output folders
            canon_folder = os.path.join(parent_folder_path, "Canon")
            crop_rgb_folder = os.path.join(parent_folder_path, "Crop_RGB")

            # Ensure the Canon folder exists
            if not os.path.exists(canon_folder):
                print(f"Skipping {parent_folder_path}, no Canon folder found.")
                continue

            # Ensure the output folder exists
            os.makedirs(crop_rgb_folder, exist_ok=True)

            # Process each image in the Canon folder
            for file_name in os.listdir(canon_folder):
                if file_name.lower().endswith(('.jpg', '.jpeg', '.png')):
                    input_path = os.path.join(canon_folder, file_name)
                    output_path = os.path.join(crop_rgb_folder, file_name)

                    # Crop and save the image
                    crop_and_save(input_path, output_path, x_start=x_start, x_end=x_end, y_start=y_start, y_end=y_end)


parent_parent_folder = "/content/drive/MyDrive/Thesis/POM-IMG/Data-12.08.2024"
x_start = 2500  # Crop pixels from the left
x_end = 7400    # Keep up to pixels from the left
y_start = 200   # Crop pixels from the top
y_end = 3500    # Keep up to pixels from the top

crop_images_in_parent_folder(parent_parent_folder, x_start=x_start, x_end=x_end, y_start=y_start, y_end=y_end)


In [ ]:
#crop thermal Tiff file before fetaire extraction
def crop_and_save_tiff(image_path, output_path, x_start=0, x_end=None, y_start=0, y_end=None):

    # Load the TIFF image (16-bit or float)
    image = tifffile.imread(image_path)

    if image is None:
        print(f"Error: Unable to load TIFF image {image_path}")
        return

    # Get image dimensions
    img_height, img_width = image.shape[:2]

    # Default x_end and y_end to the image dimensions if not specified
    if x_end is None:
        x_end = img_width
    if y_end is None:
        y_end = img_height

    # Perform the crop
    cropped_image = image[y_start:y_end, x_start:x_end]

    # Save the cropped image as a 16-bit or float TIFF
    tifffile.imwrite(output_path, cropped_image)
    print(f"Cropped TIFF saved to: {output_path}")

def crop_tiff_images_in_parent_folder(parent_parent_folder, x_start=0, x_end=None, y_start=0, y_end=None):

    for parent_folder in os.listdir(parent_parent_folder):
        parent_folder_path = os.path.join(parent_parent_folder, parent_folder)
        if os.path.isdir(parent_folder_path):
            # Locate all folders with 'thermal_tiff' in the name
            thermal_folders = [os.path.join(parent_folder_path, f) for f in os.listdir(parent_folder_path) if "thermal_tiff" in f.lower()]

            if thermal_folders:
                for idx, thermal_folder in enumerate(thermal_folders, start=1):
                    if os.path.isdir(thermal_folder):
                        crop_folder = os.path.join(parent_folder_path, f"Crop_thermal_{idx}")  # Create unique crop folder for each
                        os.makedirs(crop_folder, exist_ok=True)  # Create output folder if it doesn't exist

                        # Process each TIFF image in the thermal folder
                        for file_name in os.listdir(thermal_folder):
                            if file_name.lower().endswith(('.tiff', '.tif')):  # Only process TIFF files
                                input_path = os.path.join(thermal_folder, file_name)
                                output_path = os.path.join(crop_folder, file_name)

                                # Crop and save the TIFF image
                                crop_and_save_tiff(input_path, output_path, x_start=x_start, x_end=x_end, y_start=y_start, y_end=y_end)

parent_parent_folder = "/content/drive/MyDrive/Thesis/POM-IMG/Data-14.10.2024"
x_start = 150  # Crop pixels from the left
x_end = 375    # Keep up to pixels from the left
y_start = 190  # Crop pixels from the top
y_end = 360    # Keep up to pixels from the top

crop_tiff_images_in_parent_folder(parent_parent_folder, x_start=x_start, x_end=x_end, y_start=y_start, y_end=y_end)


In [ ]:
#Crearing motoroligcal CSV for each image in a folder
# Define the parent folder and the meteorological CSV file
parent_folder = "/content/drive/MyDrive/Thesis/POM-IMG/Data-13.08.2024"  # Adjust to your parent folder
meteorological_csv = "/content/drive/MyDrive/Thesis/POM-IMG/Moto_Data/13.08.24-Data_CLEANED.csv" # Path to the meteorological CSV

# Read the meteorological station data
meteo_data = pd.read_csv(meteorological_csv, encoding="windows-1255")  # Adjust encoding for Hebrew if necessary

# Add a formatted_time column (HHMM format) to the meteorological data for comparison
meteo_data["formatted_time"] = meteo_data["Time Aj"].str[:5].str.replace(":", "")  # Convert "05:52" -> "0552"

# List of columns to keep
columns_to_keep = [
    "Date",
    "Time Aj",
    "Temp °C Avg.",
    "Humidity % Avg.",
    "Solar Radiation W/m² Avg.",
    "Wind Speed m/sec Avg.",
    "Wind Dir Avg.",
    "TC °C Avg.",
    "Atmospheric Pressure mb Avg.",
    "Dew Point C° Avg.",
    "Temp °C Max.",
    "Humidity % Max.",
    "Solar Radiation W/m² Max.",
    "Wind Speed m/sec Max.",
    "Wind Dir",
    "TC °C Max.",
    "Dew Point Max.Max.",
    "Temp °C Min.",
    "Humidity % Min.",
    "Solar Radiation W/m² Min.",
    "Wind Speed m/sec Min.",
    "Dir",
    "TC C° Min.",
    "Dew Point Min.Min.",
    "formatted_time"  # Keep formatted_time temporarily
]
meteo_data = meteo_data[columns_to_keep]  # Keep only the specified columns

# Function to process a Canon folder
def process_canon_folder(canon_folder_path, save_folder):
    # Prepare a list to store matched rows
    matched_rows = []

    # Process each image in the Canon folder
    for image_file in os.listdir(canon_folder_path):
        if image_file.lower().endswith((".jpg", ".jpeg", ".png")):  # Check for image files
            # Extract timestamp from the filename (e.g., `0552` from `2024081205523753632IMG_0001`)
            image_time = image_file[8:12]  # Adjust slicing based on your filename structure

            # Compare with the meteorological data and find matching rows
            matched_row = meteo_data[meteo_data["formatted_time"] == image_time]
            if not matched_row.empty:
                matched_rows.append(matched_row.iloc[0].to_dict())  # Add the matched row to the list

    # Create a DataFrame from the matched rows
    if matched_rows:
        filtered_data = pd.DataFrame(matched_rows)

        # Remove the 'formatted_time' column before saving
        filtered_data = filtered_data.drop(columns=["formatted_time"])

        # Save the filtered data as a new CSV in the parent folder (e.g., `T1R1`)
        output_csv = os.path.join(save_folder, "filtered_meteorological_data.csv")
        filtered_data.to_csv(output_csv, index=False, encoding="utf-8-sig")  # Ensure proper encoding for Hebrew
        print(f"Filtered meteorological CSV saved: {output_csv}")
    else:
        print(f"No matching timestamps found for {canon_folder_path}")

# Recursively locate and process Canon folders
for root, dirs, files in os.walk(parent_folder):
    if "Canon" in dirs:  # Ensure it matches the actual subfolder name
        canon_folder_path = os.path.join(root, "Canon")
        save_folder = root  # Save in the parent folder (e.g., `T1R1`)
        print(f"Processing Canon folder: {canon_folder_path}")
        process_canon_folder(canon_folder_path, save_folder)


In [ ]:
#creating Thermal Fetures for tabular data
def extract_advanced_tiff_features(file_path):
    """Extracts metadata, statistical, texture, and frequency-based features from a thermal TIFF image."""
    try:
        # Load TIFF file
        with tifffile.TiffFile(file_path) as tif:
            thermal_image = tif.asarray()

        # Compute basic temperature statistics
        mean_temp = np.mean(thermal_image)
        std_temp = np.std(thermal_image)
        min_temp = np.min(thermal_image)
        max_temp = np.max(thermal_image)
        temp_range = max_temp - min_temp

        # Z-score normalization
        z_score_image = (thermal_image - mean_temp) / (std_temp + 1e-8)
        z_score_mean = np.mean(z_score_image)
        z_score_std = np.std(z_score_image)

        # Skewness and Kurtosis
        temp_skewness = skew(thermal_image.flatten())
        temp_kurtosis = kurtosis(thermal_image.flatten())

        # Gradient-based analysis (Sobel edges)
        sobel_x = cv2.Sobel(thermal_image, cv2.CV_64F, 1, 0, ksize=5)
        sobel_y = cv2.Sobel(thermal_image, cv2.CV_64F, 0, 1, ksize=5)
        gradient_magnitude = np.sqrt(sobel_x**2 + sobel_y**2)
        gradient_mean = np.mean(gradient_magnitude)
        gradient_std = np.std(gradient_magnitude)

        # Fourier Transform Features (frequency domain analysis)
        fft_image = np.abs(fft2(thermal_image))
        fft_mean = np.mean(fft_image)
        fft_std = np.std(fft_image)

        # Local Binary Pattern (LBP) for texture analysis
        lbp = local_binary_pattern(thermal_image, P=8, R=1, method='uniform')
        lbp_entropy = shannon_entropy(lbp)

        # Compute texture features using Haralick statistics
        thermal_image_8bit = ((thermal_image - min_temp) / (max_temp - min_temp) * 255).astype(np.uint8)
        glcm = graycomatrix(thermal_image_8bit, distances=[1], angles=[0], levels=256, symmetric=True, normed=True)
        haralick_contrast = graycoprops(glcm, 'contrast')[0, 0]
        haralick_correlation = graycoprops(glcm, 'correlation')[0, 0]
        haralick_homogeneity = graycoprops(glcm, 'homogeneity')[0, 0]
        haralick_energy = graycoprops(glcm, 'energy')[0, 0]

        # Image entropy (Shannon entropy)
        entropy = shannon_entropy(thermal_image)

        # Create a dictionary of extracted features
        extracted_features = {
            "Filename": os.path.basename(file_path),
            "MinTemperature": min_temp,
            "MaxTemperature": max_temp,
            "MeanTemperature": mean_temp,
            "StdTemperature": std_temp,
            "TemperatureRange": temp_range,
            "ZScoreMean": z_score_mean,
            "ZScoreStd": z_score_std,
            "Skewness": temp_skewness,
            "Kurtosis": temp_kurtosis,
            "GradientMean": gradient_mean,
            "GradientStd": gradient_std,
            "FFTMean": fft_mean,
            "FFTStd": fft_std,
            "LBPEntropy": lbp_entropy,
            "HaralickContrast": haralick_contrast,
            "HaralickCorrelation": haralick_correlation,
            "HaralickHomogeneity": haralick_homogeneity,
            "HaralickEnergy": haralick_energy,
            "ImageEntropy": entropy
        }

        return extracted_features

    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None

def natural_sort_key(text):
    """Sort filenames numerically, handling cases like 'frame_1.tiff' before 'frame_10.tiff'."""
    return [int(num) if num.isdigit() else num for num in re.split(r'(\d+)', text)]

def process_folders_and_save_csv(parent_folder):
    """Processes TIFF images in all subfolders, ensuring numerical order, and saves one CSV per subfolder."""
    for subfolder in sorted(os.listdir(parent_folder)):
        subfolder_path = os.path.join(parent_folder, subfolder)

        if os.path.isdir(subfolder_path):
            print(f"Processing folder: {subfolder}")

            # Find all "Crop_thermal_*" folders inside the current subfolder
            thermal_folders = [os.path.join(subfolder_path, f) for f in os.listdir(subfolder_path) if "crop_thermal" in f.lower()]
            all_features = []

            for thermal_folder in sorted(thermal_folders):
                if os.path.isdir(thermal_folder):
                    print(f"   Processing thermal folder: {thermal_folder}")

                    # Sort filenames numerically
                    tiff_files = sorted([f for f in os.listdir(thermal_folder) if f.lower().endswith(('.tiff', '.tif'))], key=natural_sort_key)

                    # Loop through TIFF files in numerical order
                    for file_name in tiff_files:
                        file_path = os.path.join(thermal_folder, file_name)
                        print(f"      Extracting features from: {file_name}")

                        # Extract features
                        features = extract_advanced_tiff_features(file_path)
                        if features:
                            all_features.append(features)

            # Save extracted features to CSV for the current subfolder
            if all_features:
                output_csv = os.path.join(subfolder_path, f"{subfolder}_thermal_features.csv")
                df = pd.DataFrame(all_features)
                df.to_csv(output_csv, index=False)
                print(f"Features saved to: {output_csv}")
            else:
                print(f"No valid images processed for {subfolder}")

# Example usage
parent_folder = "/content/drive/MyDrive/Thesis/POM-IMG/Data-14.10.2024"  # Parent folder containing subfolders
process_folders_and_save_csv(parent_folder)

In [ ]:
#temporal merge moto tabular data with thermal tabular DATA
def clean_column_names(df):
    """Removes unwanted Unicode characters and standardizes column names."""
    df.columns = df.columns.astype(str).str.strip()
    return df

def rename_columns(merged_df):
    """Forcefully renames specific columns after merging to match the correct format."""
    rename_dict = {
        "Temp ֲ°C Avg.": "Temp °C Avg.",
        "Humidity % Avg.": "Humidity % Avg.",
        "Solar Radiation W/mֲ² Avg.": "Solar Radiation W/m² Avg.",
        "Wind Speed m/sec Avg.": "Wind Speed m/sec Avg.",
        "Wind Dir Avg.": "Wind Dir Avg.",
        "TC ֲ°C Avg.": "TC °C Avg.",
        "Atmospheric Pressure mb Avg.": "Atmospheric Pressure mb Avg.",
        "Dew Point Cֲ° Avg.": "Dew Point C° Avg.",
        "TC Cֲ° Min.": "TC C° Min.",
        "Solar Radiation W/mֲ² Max.": "Solar Radiation W/m² Max.",
        "Solar Radiation W/mֲ² Min.": "Solar Radiation W/m² Min."
    }

    merged_df = merged_df.rename(columns=rename_dict)
    return merged_df

def merge_data_in_subfolders(parent_folder):
    """Merges thermal and meteorological CSV files in each subfolder, ensuring clean column names and correct alignment."""

    for subfolder in sorted(os.listdir(parent_folder)):
        subfolder_path = os.path.join(parent_folder, subfolder)

        if os.path.isdir(subfolder_path):  # Only process directories
            print(f"Processing folder: {subfolder}")

            # Find thermal and meteorological CSV files
            thermal_csv = None
            meteorological_csv = None

            for file in os.listdir(subfolder_path):
                if file.endswith("_thermal_features.csv"):
                    thermal_csv = os.path.join(subfolder_path, file)
                elif file.startswith("filtered_meteorological_data"):
                    meteorological_csv = os.path.join(subfolder_path, file)

            # Proceed if both files exist
            if thermal_csv and meteorological_csv:
                print(f"   Found thermal CSV: {thermal_csv}")
                print(f"   Found meteorological CSV: {meteorological_csv}")

                # Load datasets
                thermal_df = pd.read_csv(thermal_csv, encoding='utf-8-sig')
                meteorological_df = pd.read_csv(meteorological_csv, encoding='utf-8-sig')

                # Clean column names to remove unwanted symbols
                thermal_df = clean_column_names(thermal_df)
                meteorological_df = clean_column_names(meteorological_df)

                # Drop only specified Min/Max columns from meteorological data
                meteorological_df = meteorological_df.drop(columns=[
                    "Temp °C Min.", "Temp °C Max.",
                    "Humidity % Min.", "Humidity % Max.", "Solar Radiation W/m² Min.", "Solar Radiation W/m² Max.",
                    "Wind Speed m/sec Min.", "Wind Speed m/sec Max.", "Wind Dir Min.", "Wind Dir Max.",
                    "TC °C Min.", "TC °C Max.", "Dew Point Min.", "Dew Point Max.", "Dew Point Min.Min.", "Dew Point Max.Max."
                ], errors="ignore")

                # Drop only the specified columns from thermal data (do not remove MeanTemperature!)
                thermal_df = thermal_df.drop(columns=["MinTemperature", "MaxTemperature", "Filename"], errors="ignore")

                # Keep only data from row 1 onward in thermal dataset
                thermal_df = thermal_df.iloc[1:, :].reset_index(drop=True)

                # Ensure both datasets have the same number of rows
                min_rows = min(meteorological_df.shape[0], thermal_df.shape[0])
                meteorological_df = meteorological_df.iloc[:min_rows, :].reset_index(drop=True)
                thermal_df = thermal_df.iloc[:min_rows, :].reset_index(drop=True)

                # Merge datasets (stop where meteorological data ends)
                merged_df = pd.concat([meteorological_df, thermal_df], axis=1)

                # Rename columns AFTER merging
                merged_df = rename_columns(merged_df)

                # Save merged data in the same subfolder
                merged_output_path = os.path.join(subfolder_path, "merged_data.csv")
                merged_df.to_csv(merged_output_path, index=False, encoding='utf-8-sig')
                print(f" Merged data saved to: {merged_output_path}")

            else:
                print(f" Missing thermal or meteorological CSV in {subfolder}")

# Example usage
parent_folder = "/content/drive/MyDrive/Thesis/POM-IMG/Data-24.09.2024"  # Update with correct path
merge_data_in_subfolders(parent_folder)
